# HemaVision — Uncertainty-Aware Anemia Triage
AMD Developer Hackathon: ACT II — Unicorn Track

**Pipeline:** Conjunctiva image → MobileNetV2 + MC Dropout (uncertainty) → Fireworks AI (gpt-oss-20b) reasoning → Triage report

Run all cells top to bottom, in order.

## 1. Setup — Mount Drive & Install Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install fastapi uvicorn tensorflow pillow requests python-multipart pyngrok --break-system-packages -q

Mounted at /content/drive


## 2. Configuration — set your paths and keys here (ONLY place you edit)

In [ ]:
from getpass import getpass
import os

# EDIT THESE THREE VALUES
MODEL_PATH = "/content/drive/MyDrive/anemia_project/models/mobilenet_final.keras"
TEST_IMAGE_PATH = "/content/drive/MyDrive/Combined_Conjunctiva_Dataset/labeled/masks/India_11_20200203_094523_palpebral.png"
NGROK_AUTHTOKEN = getpass("auth token from ngrok.com/auth:")
FIREWORKS_API_KEY = getpass("fireworks key: ")

# Sanity checks
print("Model file exists:", os.path.exists(MODEL_PATH))
print("Test image exists:", os.path.exists(TEST_IMAGE_PATH))

os.environ["MODEL_PATH"] = MODEL_PATH
os.environ["FIREWORKS_API_KEY"] = FIREWORKS_API_KEY

auth token from ngrok.com/auth:··········
fireworks key: ··········
Model file exists: True
Test image exists: True


## 3. Write the FastAPI backend (main.py)
Combines MC Dropout uncertainty model + Fireworks (gpt-oss-20b) reasoning layer.

In [ ]:
%%writefile main.py
import os
import io
import json
import requests
import numpy as np
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from PIL import Image

import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

MODEL_PATH = os.environ.get("MODEL_PATH", "mobilenet_final.keras")
FIREWORKS_API_KEY = os.environ.get("FIREWORKS_API_KEY", "")
FIREWORKS_URL = "https://api.fireworks.ai/inference/v1/chat/completions"
FIREWORKS_MODEL = "accounts/fireworks/models/gpt-oss-20b"

# Calibrated on a small validation sample (75th percentile of MC Dropout std_dev).
# NOTE: recalibrate on a larger validation set before production use.
UNCERTAINTY_STD_THRESHOLD = 0.095
MC_DROPOUT_ITERATIONS = 30

app = FastAPI(title="HemaVision Triage API")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

model = None

@app.on_event("startup")
def load_the_model():
    global model
    if not os.path.exists(MODEL_PATH):
        print(f"WARNING: model file not found at {MODEL_PATH}")
        return
    model = load_model(MODEL_PATH)
    print("Model loaded successfully.")

def preprocess_image_bytes(image_bytes: bytes, target_size=(224, 224)) -> np.ndarray:
    img = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    img = img.resize(target_size)
    arr = np.array(img).astype("float32")
    arr = np.expand_dims(arr, axis=0)
    arr = preprocess_input(arr)
    return arr

def mc_dropout_predict(model, image_array, n_iterations=MC_DROPOUT_ITERATIONS):
    predictions = []
    for _ in range(n_iterations):
        pred = model(image_array, training=True)
        predictions.append(float(pred.numpy().flatten()[0]))
    predictions = np.array(predictions)
    return float(predictions.mean()), float(predictions.std()), predictions.tolist()

def classify_with_uncertainty(mean_prob, std_dev, std_threshold=UNCERTAINTY_STD_THRESHOLD):
    is_uncertain = std_dev > std_threshold
    if mean_prob < 0.35:
        risk = "Low"
    elif mean_prob < 0.65:
        risk = "Moderate"
    else:
        risk = "High"
    return {
        "predicted_probability": round(mean_prob, 4),
        "uncertainty_std": round(std_dev, 4),
        "confident_prediction": not is_uncertain,
        "risk_tier": risk,
    }

def get_llm_reasoning(model_result: dict, age: str, gender: str, symptoms: str) -> dict:
    if not FIREWORKS_API_KEY:
        raise HTTPException(status_code=500, detail="FIREWORKS_API_KEY not set on server")

    system_prompt = (
        "You are a medical triage assistant supporting a non-invasive anemia "
        "screening tool. This tool is for SCREENING and EDUCATION only \u2014 it is "
        "NOT a diagnosis and must not replace laboratory hemoglobin testing. "
        "Respond ONLY with a valid JSON object, no other text, no reasoning shown."
    )

    user_prompt = f"""
    Image-based anemia screening result (MobileNetV2 + MC Dropout uncertainty):
    - predicted_probability: {model_result['predicted_probability']} (probability of anemia, 0-1 scale)
    - uncertainty_std: {model_result['uncertainty_std']} (std dev across 30 stochastic forward passes)
    - confident_prediction: {model_result['confident_prediction']}
    - risk_tier: {model_result['risk_tier']}

    Patient-reported context:
    - age: {age or "not provided"}
    - gender: {gender or "not provided"}
    - symptoms: {symptoms or "none reported"}

    Return a JSON object with exactly these keys:
    - risk_level (string: Low/Moderate/High)
    - explanation (2-3 sentences, plain simple language, non-diagnostic tone)
    - recommended_tests (array of strings, e.g. CBC, ferritin)
    - nutrition_advice (1-2 sentences, general iron-rich diet guidance)
    - referral_urgency (string: low/moderate/urgent)
    - disclaimer (string: a short reminder this is a screening tool, not a diagnosis)

    If confident_prediction is false, explicitly mention in the explanation that
    the model itself is uncertain about this image and a repeat test or clinical
    exam is recommended regardless of the risk tier.
    """

    payload = {
        "model": FIREWORKS_MODEL,
        "max_tokens": 1000,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    }
    headers = {
        "Authorization": f"Bearer {FIREWORKS_API_KEY}",
        "Content-Type": "application/json",
    }

    response = requests.post(FIREWORKS_URL, headers=headers, json=payload, timeout=30)
    if response.status_code != 200:
        raise HTTPException(status_code=502, detail=f"Fireworks API error ({response.status_code}): {response.text}")

    data = response.json()
    raw_content = data["choices"][0]["message"]["content"]
    try:
        return json.loads(raw_content)
    except json.JSONDecodeError:
        cleaned = raw_content.strip().strip("```json").strip("```").strip()
        try:
            return json.loads(cleaned)
        except json.JSONDecodeError:
            raise HTTPException(status_code=502, detail=f"LLM did not return valid JSON: {raw_content[:300]}")

@app.get("/health")
def health():
    return {
        "status": "ok" if model is not None else "model_not_loaded",
        "model_path": MODEL_PATH,
        "fireworks_key_configured": bool(FIREWORKS_API_KEY),
    }

@app.post("/analyze")
async def analyze(
    image: UploadFile = File(...),
    age: str = Form(default=""),
    gender: str = Form(default=""),
    symptoms: str = Form(default=""),
):
    if model is None:
        raise HTTPException(status_code=503, detail="Model not loaded on server")

    image_bytes = await image.read()
    try:
        image_array = preprocess_image_bytes(image_bytes)
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"Invalid image file: {e}")

    mean_prob, std_dev, all_preds = mc_dropout_predict(model, image_array)
    model_result = classify_with_uncertainty(mean_prob, std_dev)
    llm_result = get_llm_reasoning(model_result, age, gender, symptoms)

    return {
        "model_analysis": model_result,
        "reasoning": llm_result,
    }

Writing main.py


## 4. Start the server (ngrok + uvicorn)

In [ ]:
from pyngrok import ngrok
import subprocess, time

# Kill any previously running server (safe to run every time)
!pkill -f uvicorn
ngrok.kill()
time.sleep(2)

ngrok.set_auth_token(NGROK_AUTHTOKEN)

process = subprocess.Popen(["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"])
time.sleep(6)

tunnel = ngrok.connect(8000)
public_url = tunnel.public_url
print("Public URL:", public_url)

Public URL: https://frugality-bonehead-mortally.ngrok-free.dev


## 5. Test — Health Check

In [ ]:
import requests

r = requests.get(f"{public_url}/health", timeout=10)
print("Status:", r.status_code)
try:
    print(r.json())
except requests.exceptions.JSONDecodeError:
    print("Response was not JSON. Raw response content:")
    print(r.text)

Status: 200
{'status': 'ok', 'model_path': '/content/drive/MyDrive/anemia_project/models/mobilenet_final.keras', 'fireworks_key_configured': True}


## 6. Test — Full Analyze Endpoint (image + symptoms → triage report)

In [ ]:
files = {"image": open(TEST_IMAGE_PATH, "rb")}
data = {"age": "25", "gender": "female", "symptoms": "fatigue,dizziness"}

response = requests.post(f"{public_url}/analyze", files=files, data=data, timeout=60)
files["image"].close()

print("Status Code:", response.status_code)
print("Response JSON:")
import json
print(json.dumps(response.json(), indent=2))

Status Code: 200
Response JSON:
{
  "model_analysis": {
    "predicted_probability": 0.6821,
    "uncertainty_std": 0.1123,
    "confident_prediction": false,
    "risk_tier": "High"
  },
  "reasoning": {
    "risk_level": "High",
    "explanation": "The tool estimates that there is about a 68% chance of anemia based on the image, but it is not confident in this result. Because the model is uncertain, we recommend a repeat test or a clinical examination regardless of the risk tier. It\u2019s important to have additional information before making any decisions.",
    "recommended_tests": [
      "CBC",
      "Ferritin",
      "Iron studies"
    ],
    "nutrition_advice": "Eating iron-rich foods such as lean red meat, beans, lentils, and leafy green vegetables can support healthy blood levels. Pair these foods with vitamin C\u2011rich items like citrus or tomatoes to help your body absorb iron better.",
    "referral_urgency": "urgent",
    "disclaimer": "This is a screening tool and not